# Engagement notebook

This is the original analysis notebook, pointed at data the project actually has.

It used to load `data/your_data.csv` and a placeholder column named `column_name`, so the cells could not run. It now reads `data/exports/engagement.csv`. `python scripts/build_dashboard.py` writes that file from the same snapshots as the dashboard (or from the labeled demo set when no live snapshots exist).

`data_source` is `demo` or `live`. Demo rows are sample numbers. They are not your real stats.

The pass below matches the original template: dataset shape, missing values, duplicates, a simple outlier screen, the engagement distribution, a correlation heatmap, and engagement by category.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set(style="whitegrid")

candidates = [
    Path("../data/exports/engagement.csv"),
    Path("data/exports/engagement.csv"),
]
path = next((item for item in candidates if item.exists()), None)
if path is None:
    raise FileNotFoundError(
        "Missing data/exports/engagement.csv. From the repo root run: python scripts/build_dashboard.py"
    )

data = pd.read_csv(path)
print("Loaded", path)
print("\nDataset Info:")
data.info()
print("\nStatistical Summary:")
print(data.describe(include="all"))
print("\nMissing Values Count:")
missing_values = data.isnull().sum()
print(missing_values[missing_values > 0] if (missing_values > 0).any() else "None")
print("\nRows by data_source:")
print(data["data_source"].value_counts().to_string())
if set(data["data_source"].dropna().unique()) == {"demo"}:
    print("\nDEMO DATA: these rows are samples so the notebook can run before API keys exist.")
print("\nBlank shares are expected for YouTube. The public API does not provide a share count.")


In [ ]:
before = len(data)
data = data.drop_duplicates()
print(f"Duplicate rows removed: {before - len(data)}. Rows left: {len(data)}.")

# YouTube share cells are blank on purpose. Leave them blank.
# Fill only engagement when a row somehow lost that total.
if data["engagement"].isna().any():
    data["engagement"] = data["engagement"].fillna(data["engagement"].median())

engagement = data["engagement"]
q1 = engagement.quantile(0.25)
q3 = engagement.quantile(0.75)
iqr = q3 - q1
print(f"Engagement IQR: {iqr:.1f} (Q1={q1:.1f}, Q3={q3:.1f})")
if iqr > 0:
    outlier = (engagement < (q1 - 1.5 * iqr)) | (engagement > (q3 + 1.5 * iqr))
    filtered = data.loc[~outlier]
    print(f"Outlier screen flagged {int(outlier.sum())} rows.")
    if len(filtered) >= 5:
        data = filtered.copy()
        print(f"Charts below use {len(data)} rows after that screen.")
    else:
        print("Kept every row so the charts still have enough posts.")
else:
    print("IQR is 0, so no outlier screen was applied.")


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data["engagement"], bins=15, kde=True)
plt.title("Distribution of engagement (likes + comments + shares)")
plt.xlabel("Engagement")
plt.ylabel("Posts")
plt.tight_layout()
plt.show()

numbers = data[["engagement", "likes", "comments", "shares", "views"]].apply(pd.to_numeric, errors="coerce")
plt.figure(figsize=(8, 6))
sns.heatmap(numbers.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation of engagement counts")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(x="category", y="engagement", data=data)
plt.title("Engagement by category")
plt.xlabel("Category")
plt.ylabel("Engagement")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
by_category = data.groupby("category", observed=True)["engagement"].median().sort_values(ascending=False)
print("Median engagement by category:")
print(by_category.to_string())
print()
top_category = by_category.index[0]
print(
    f"The highest median engagement in this export is '{top_category}' "
    f"at {by_category.iloc[0]:.0f}."
)
print(
    "Title length, posting hour, and rising keywords are computed on the dashboard "
    "in the 'What to do better' section, from the full snapshot history."
)
